# Metadata

> Metadata introspection for the Silero VAD plugin used by cjm-ctl to generate the registration manifest.

In [ ]:
#| default_exp meta

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
import sys
from typing import Any, Dict
from cjm_media_plugin_silero_vad import __version__

In [ ]:
#| export
def get_plugin_metadata() -> Dict[str, Any]:  # Plugin metadata for manifest generation
    """Return metadata required to register this plugin with the PluginManager."""
    # Fallback base path (current behavior for backward compatibility)
    base_path = os.path.dirname(os.path.dirname(sys.executable))
    
    # Use CJM config if available
    cjm_data_dir = os.environ.get("CJM_DATA_DIR")
    
    # Plugin data directory
    plugin_name = "cjm-media-plugin-silero-vad"
    if cjm_data_dir:
        data_dir = os.path.join(cjm_data_dir, plugin_name)
    else:
        data_dir = os.path.join(base_path, "data")
    
    db_path = os.path.join(data_dir, "vad_jobs.db")
    
    # Ensure data directory exists
    os.makedirs(data_dir, exist_ok=True)

    return {
        "name": plugin_name,
        "version": __version__,
        # T24: non-empty description required by the substrate validator (SG-6 / V1 gate).
        "description": "Voice Activity Detection (speech segmentation) via Silero VAD with SQLite result caching.",
        "type": "media-analysis",
        "category": "media",
        
        # Connects to the Tier 2 Interface
        "interface": "cjm_media_plugin_system.analysis_interface.MediaAnalysisPlugin",
        
        "module": "cjm_media_plugin_silero_vad.plugin",
        "class": "SileroVADPlugin",
        
        # Critical: The absolute path to THIS environment's python
        "python_path": sys.executable,
        
        "db_path": db_path,
        
        # Phase 5a / CR-7 reframe: binary hard-facts only (quantitative resource
        # amounts dropped — the substrate measures empirically, V12 gate).
        "resources": {
            "requires_gpu": False
        },
        
        # Track 19: visible worker env is now declared on the plugin class via
        # WORKER_ENV (EnvVarSpec); the substrate composes + injects it at spawn.
        "env_vars": {}
    }

## Testing

In [ ]:
import json

metadata = get_plugin_metadata()
print(json.dumps(metadata, indent=2))

{
  "name": "cjm-media-plugin-silero-vad",
  "version": "0.0.1",
  "type": "media-analysis",
  "category": "media",
  "interface": "cjm_media_plugin_system.analysis_interface.MediaAnalysisPlugin",
  "module": "cjm_media_plugin_silero_vad.plugin",
  "class": "SileroVADPlugin",
  "python_path": "/home/innom-dt/miniforge3/envs/cjm-media-plugin-silero-vad/bin/python3.11",
  "db_path": "/home/innom-dt/miniforge3/envs/cjm-media-plugin-silero-vad/data/vad_jobs.db",
  "resources": {
    "requires_gpu": false,
    "min_system_ram_mb": 4096
  },
  "env_vars": {
    "OMP_NUM_THREADS": "4"
  }
}


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()